# Kaptia — iPhone 可操作的雲端 QLoRA 訓練

在 Google Colab 中開啟此 Notebook，先於 **Runtime → Change runtime type** 選擇 GPU。免費 GPU 不保證供應、記憶體或工作階段時長；若中斷，請從頭檢查輸出與 checkpoint，不要假設訓練成功。

本流程微調 `HuggingFaceTB/SmolLM2-1.7B-Instruct` 的 adapter，不是從零預訓練完整模型。請先確認資料使用權與 Hugging Face 模型授權。

In [ ]:
import os, subprocess, sys, torch
print('Python:', sys.version)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    raise RuntimeError('尚未取得 GPU。請在 Colab Runtime 設定中選擇 GPU，然後重新執行本格。')

In [ ]:
REPO_URL = 'https://github.com/argusocean-wq/kaptia.git'
if not os.path.isdir('/content/kaptia/.git'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, '/content/kaptia'], check=True)
%cd /content/kaptia
!git rev-parse HEAD
!ls training

In [ ]:
# Install project training dependencies. This can take several minutes.
!python -m pip install --upgrade pip
!python -m pip install -r requirements-train.txt

In [ ]:
# Preflight and dataset checks; stop if any check fails.
!python scripts/train_preflight.py
!python scripts/validate_training_data.py
!python scripts/validate_code_dataset.py

## 開始訓練

先使用 1 epoch、512 tokens 做小規模試跑，確認 GPU 記憶體與依賴可用。通過後再考慮增加到 2 epochs / 1024 tokens。這是建議的起始設定，不代表品質提升。

In [ ]:
os.environ['KAPTIA_EPOCHS'] = '1'
os.environ['KAPTIA_MAX_LENGTH'] = '512'
os.environ['KAPTIA_BATCH_SIZE'] = '1'
os.environ['KAPTIA_GRAD_ACCUM_STEPS'] = '8'
os.environ['PYTHONUNBUFFERED'] = '1'
subprocess.run([sys.executable, 'scripts/train_qlora.py'], check=True)

In [ ]:
# Confirm expected adapter and metrics exist before evaluating.
from pathlib import Path
adapter = Path('outputs/kaptia-smollm2-qlora/adapter')
print('adapter exists:', adapter.is_dir())
print('adapter config exists:', (adapter / 'adapter_config.json').is_file())
print('metrics exists:', Path('outputs/kaptia-smollm2-qlora/metrics.json').is_file())
if not (adapter / 'adapter_config.json').is_file():
    raise FileNotFoundError('找不到可辨識的 adapter；停止評測，請檢查訓練日誌。')

In [ ]:
# Compare base model and adapter on held-out prompts.
subprocess.run([sys.executable, 'scripts/evaluate_adapter.py'], check=True)
!ls -lh outputs/kaptia-smollm2-qlora

In [ ]:
# Package adapter + metrics + comparison for download to iPhone.
from pathlib import Path
import zipfile
bundle = Path('/content/kaptia-results.zip')
include = [Path('outputs/kaptia-smollm2-qlora/adapter'), Path('outputs/kaptia-smollm2-qlora/metrics.json'), Path('outputs/kaptia-smollm2-qlora/comparison.jsonl')]
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for item in include:
        if item.is_dir():
            for f in item.rglob('*'):
                if f.is_file(): z.write(f, f.as_posix())
        elif item.is_file():
            z.write(item, item.as_posix())
print('Bundle:', bundle, 'size MiB:', round(bundle.stat().st_size/1024**2, 2))
from google.colab import files
files.download(str(bundle))

## 結果判讀與限制

- Notebook 每一格都需由使用者手動執行；不能從 GitHub 自動取得 Colab GPU。
- `comparison.jsonl` 是基礎模型與 adapter 的回答並列，不是自動品質分數。請人工逐題評分並記錄錯誤。
- 訓練輸出目前只保存在 Colab 執行環境；工作階段結束前請下載 ZIP。若工作階段中斷，未下載資料可能遺失。
- 請勿把 token、API 金鑰或私人資料寫入 Notebook 或公開倉庫。
- 這個流程不會把 adapter 自動部署到 iPhone Safari，也不代表 Kaptia 已完成端到端驗收。